# Data Profiling

**Objective**: This notebook executes the initial Data Profiling on a local instance of the raw data. The goal here is not to transform the data, but to analyze its structure, identify anomalies (e.g., operational product codes like 'POST' or 'M'), and map missing values. The insights gathered during this exploratory phase dictate the data cleaning rules and define the architecture of the Star Schema deployed on Google BigQuery.

**Architectural Choice**: In a real-world Big Data scenario, local exploration would be hindered by RAM limits, requiring aggregate query strategies (Metadata Profiling) directly on the Data Lake via serverless technologies. However, since this specific dataset falls within the *Small Data* spectrum, the most efficient and cost-effective approach is full *In-Memory Processing* using local Pandas. This allows for rapid and comprehensive Data Discovery on the entire dataset population before defining the Star Schema constraints and shifting the computational workload (ETL and Machine Learning) to the Google Cloud infrastructure.

In [1]:
import pandas as pd
import os

# Navigate up one level to exit 'notebooks' and access the 'data' folder
path_raw = os.path.join('data', 'online_retail_raw.csv')
df = pd.read_csv(path_raw)

print(f"Rows loaded for analysis: {len(df)}")

Rows loaded for analysis: 541909


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [3]:
df.describe(include='all')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900,4070,4223,NaN,23260,NaN,NaN,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,10/31/2011 14:41,NaN,NaN,United Kingdom
freq,1114,2313,2369,NaN,1114,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,NaN,4.611114,15287.690570,NaN
std,NaN,NaN,NaN,218.081158,NaN,96.759853,1713.600303,NaN
min,NaN,NaN,NaN,-80995.000000,NaN,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,NaN,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,NaN,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,NaN,4.130000,16791.000000,NaN


# InvoiceNo

In [4]:
unique_invoices = len(df['InvoiceNo'].unique())
print(f"Unique invoices: {unique_invoices} out of {len(df)} rows ({unique_invoices / len(df) * 100:.2f}% of the dataset)")

Unique invoices: 25900 out of 541909 rows (4.78% of the dataset)


In [5]:
display(df.groupby('InvoiceNo').agg({'StockCode': 'count'}).sort_values('StockCode', ascending=False).head())

,StockCode
InvoiceNo,
573585,1114
581219,749
581492,731
580729,721
558475,705


In [6]:
print(f"Cancellations (InvoiceNo starts with C): {len(df[df['InvoiceNo'].str.startswith('C', na=False)])}")

Cancellations (InvoiceNo starts with C): 9288


# StockCode

In [7]:
unique_stockcodes = len(df['StockCode'].unique())
print(f"Unique StockCodes: {unique_stockcodes} out of {len(df)} rows ({unique_stockcodes / len(df) * 100:.2f}% of the dataset)")

Unique StockCodes: 4070 out of 541909 rows (0.75% of the dataset)


### Special Stock Codes Dictionary:
* M
* D
* POST
* DOT
* CRUK
* B
* BANK CHARGES
* AMAZONFEE
* S

In [8]:
def getInfoStockCode(value):
    subset = df[df['StockCode'] == value]
    display(subset.head())
    
    total_matches = len(subset)
    print(f"Unique StockCodes matching '{value}': {total_matches} out of {len(df)} ({total_matches / len(df) * 100:.2f}% of dataset)")
    print("Breakdown:")
    print(f"  * {len(subset[subset['UnitPrice'] == 0])} have UnitPrice equal to 0")
    print(f"  * {len(subset[subset['UnitPrice'] < 0])} have negative UnitPrice")
    print(f"  * {len(subset[subset['Quantity'] == 0])} have Quantity equal to 0")
    print(f"  * {len(subset[subset['Quantity'] < 0])} have negative Quantity")
    print(f"  * {len(subset[subset['CustomerID'].isnull()])} have null CustomerID")
    print(f"  * {len(subset[subset['InvoiceNo'].str.startswith('C', na=False)])} have an InvoiceNo starting with 'C' (Cancellations/Returns)")

## StockCode with M

In [9]:
getInfoStockCode('M')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2239,536569,M,Manual,1,12/1/2010 15:35,1.25,16274.0,United Kingdom
2250,536569,M,Manual,1,12/1/2010 15:35,18.95,16274.0,United Kingdom
5684,536865,M,Manual,1,12/3/2010 11:28,2.55,NaN,United Kingdom
6798,536981,M,Manual,2,12/3/2010 14:26,0.85,14723.0,United Kingdom
7976,537077,M,Manual,12,12/5/2010 11:59,0.42,17062.0,United Kingdom


Unique StockCodes matching 'M': 571 out of 541909 (0.11% of dataset)
Breakdown:
  * 6 have UnitPrice equal to 0
  * 0 have negative UnitPrice
  * 0 have Quantity equal to 0
  * 244 have negative Quantity
  * 106 have null CustomerID
  * 244 have an InvoiceNo starting with 'C' (Cancellations/Returns)


## StockCode with D

In [10]:
getInfoStockCode('D')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,12/1/2010 9:41,27.50,14527.0,United Kingdom
9038,C537164,D,Discount,-1,12/5/2010 13:21,29.29,14527.0,United Kingdom
14498,C537597,D,Discount,-1,12/7/2010 12:34,281.00,15498.0,United Kingdom
19392,C537857,D,Discount,-1,12/8/2010 16:00,267.12,17340.0,United Kingdom
31134,C538897,D,Discount,-1,12/15/2010 9:14,5.76,16422.0,United Kingdom


Unique StockCodes matching 'D': 77 out of 541909 (0.01% of dataset)
Breakdown:
  * 0 have UnitPrice equal to 0
  * 0 have negative UnitPrice
  * 0 have Quantity equal to 0
  * 77 have negative Quantity
  * 0 have null CustomerID
  * 77 have an InvoiceNo starting with 'C' (Cancellations/Returns)


> [!note]
> All 'Discount' records feature negative quantities.

## StockCode with POST

In [11]:
getInfoStockCode('POST')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
45,536370,POST,POSTAGE,3,12/1/2010 8:45,18.0,12583.0,France
386,536403,POST,POSTAGE,1,12/1/2010 11:27,15.0,12791.0,Netherlands
1123,536527,POST,POSTAGE,1,12/1/2010 13:04,18.0,12662.0,Germany
5073,536840,POST,POSTAGE,1,12/2/2010 18:27,18.0,12738.0,Germany
5258,536852,POST,POSTAGE,1,12/3/2010 9:51,18.0,12686.0,France


Unique StockCodes matching 'POST': 1256 out of 541909 (0.23% of dataset)
Breakdown:
  * 4 have UnitPrice equal to 0
  * 0 have negative UnitPrice
  * 0 have Quantity equal to 0
  * 126 have negative Quantity
  * 60 have null CustomerID
  * 126 have an InvoiceNo starting with 'C' (Cancellations/Returns)


## StockCode with DOT

In [12]:
getInfoStockCode('DOT')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
1814,536544,DOT,DOTCOM POSTAGE,1,12/1/2010 14:32,569.77,NaN,United Kingdom
3041,536592,DOT,DOTCOM POSTAGE,1,12/1/2010 17:06,607.49,NaN,United Kingdom
5450,536862,DOT,DOTCOM POSTAGE,1,12/3/2010 11:13,254.43,NaN,United Kingdom
5545,536864,DOT,DOTCOM POSTAGE,1,12/3/2010 11:27,121.06,NaN,United Kingdom
5685,536865,DOT,DOTCOM POSTAGE,1,12/3/2010 11:28,498.47,NaN,United Kingdom


Unique StockCodes matching 'DOT': 710 out of 541909 (0.13% of dataset)
Breakdown:
  * 3 have UnitPrice equal to 0
  * 0 have negative UnitPrice
  * 0 have Quantity equal to 0
  * 1 have negative Quantity
  * 694 have null CustomerID
  * 1 have an InvoiceNo starting with 'C' (Cancellations/Returns)


## StockCode with CRUK

In [13]:
getInfoStockCode('CRUK')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
317508,C564763,CRUK,CRUK Commission,-1,8/30/2011 10:49,1.60,14096.0,United Kingdom
324023,C565382,CRUK,CRUK Commission,-1,9/2/2011 15:45,13.01,14096.0,United Kingdom
333779,C566216,CRUK,CRUK Commission,-1,9/9/2011 15:17,15.96,14096.0,United Kingdom
338848,C566565,CRUK,CRUK Commission,-1,9/13/2011 12:32,52.24,14096.0,United Kingdom
351003,C567655,CRUK,CRUK Commission,-1,9/21/2011 14:40,608.66,14096.0,United Kingdom


Unique StockCodes matching 'CRUK': 16 out of 541909 (0.00% of dataset)
Breakdown:
  * 0 have UnitPrice equal to 0
  * 0 have negative UnitPrice
  * 0 have Quantity equal to 0
  * 16 have negative Quantity
  * 0 have null CustomerID
  * 16 have an InvoiceNo starting with 'C' (Cancellations/Returns)


> [!note]
> All 'CRUK' records feature negative quantities.

## StockCode with B

In [14]:
getInfoStockCode('B')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299982,A563185,B,Adjust bad debt,1,8/12/2011 14:50,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,8/12/2011 14:51,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,8/12/2011 14:52,-11062.06,NaN,United Kingdom


Unique StockCodes matching 'B': 3 out of 541909 (0.00% of dataset)
Breakdown:
  * 0 have UnitPrice equal to 0
  * 2 have negative UnitPrice
  * 0 have Quantity equal to 0
  * 0 have negative Quantity
  * 3 have null CustomerID
  * 0 have an InvoiceNo starting with 'C' (Cancellations/Returns)


## StockCode with BANK CHARGES

In [15]:
getInfoStockCode('BANK CHARGES')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
4406,536779,BANK CHARGES,Bank Charges,1,12/2/2010 15:08,15.00,15823.0,United Kingdom
14435,C537572,BANK CHARGES,Bank Charges,-1,12/7/2010 12:00,95.38,NaN,United Kingdom
28992,C538680,BANK CHARGES,Bank Charges,-1,12/13/2010 17:10,966.92,NaN,United Kingdom
62508,541505,BANK CHARGES,Bank Charges,1,1/18/2011 15:58,15.00,15939.0,United Kingdom
64573,C541653,BANK CHARGES,Bank Charges,-1,1/20/2011 11:50,1050.15,NaN,United Kingdom


Unique StockCodes matching 'BANK CHARGES': 37 out of 541909 (0.01% of dataset)
Breakdown:
  * 0 have UnitPrice equal to 0
  * 0 have negative UnitPrice
  * 0 have Quantity equal to 0
  * 25 have negative Quantity
  * 25 have null CustomerID
  * 25 have an InvoiceNo starting with 'C' (Cancellations/Returns)


In [16]:
df[(df['StockCode'] == 'BANK CHARGES') & (df['Quantity'] > 0)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
4406,536779,BANK CHARGES,Bank Charges,1,12/2/2010 15:08,15.000,15823.0,United Kingdom
62508,541505,BANK CHARGES,Bank Charges,1,1/18/2011 15:58,15.000,15939.0,United Kingdom
152966,549717,BANK CHARGES,Bank Charges,1,4/11/2011 14:56,15.000,14606.0,United Kingdom
175275,551945,BANK CHARGES,Bank Charges,1,5/5/2011 11:09,15.000,16714.0,United Kingdom
327921,565735,BANK CHARGES,Bank Charges,1,9/6/2011 12:25,15.000,16904.0,United Kingdom
361740,568375,BANK CHARGES,Bank Charges,1,9/26/2011 17:01,15.000,13405.0,United Kingdom
361741,568375,BANK CHARGES,Bank Charges,1,9/26/2011 17:01,0.001,13405.0,United Kingdom
407618,571900,BANK CHARGES,Bank Charges,1,10/19/2011 14:26,15.000,13263.0,United Kingdom
431351,573586,BANK CHARGES,Bank Charges,1,10/31/2011 14:48,15.000,14704.0,United Kingdom
440745,574546,BANK CHARGES,Bank Charges,1,11/4/2011 14:59,15.000,13651.0,United Kingdom


## StockCode with AMAZONFEE

In [17]:
getInfoStockCode('AMAZONFEE')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
14514,C537600,AMAZONFEE,AMAZON FEE,-1,12/7/2010 12:41,1.00,NaN,United Kingdom
15016,C537630,AMAZONFEE,AMAZON FEE,-1,12/7/2010 15:04,13541.33,NaN,United Kingdom
15017,537632,AMAZONFEE,AMAZON FEE,1,12/7/2010 15:08,13541.33,NaN,United Kingdom
16232,C537644,AMAZONFEE,AMAZON FEE,-1,12/7/2010 15:34,13474.79,NaN,United Kingdom
16313,C537647,AMAZONFEE,AMAZON FEE,-1,12/7/2010 15:41,5519.25,NaN,United Kingdom


Unique StockCodes matching 'AMAZONFEE': 34 out of 541909 (0.01% of dataset)
Breakdown:
  * 0 have UnitPrice equal to 0
  * 0 have negative UnitPrice
  * 0 have Quantity equal to 0
  * 32 have negative Quantity
  * 34 have null CustomerID
  * 32 have an InvoiceNo starting with 'C' (Cancellations/Returns)


In [18]:
df[(df['StockCode'] == 'AMAZONFEE') & (df['Quantity'] > 0)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
15017,537632,AMAZONFEE,AMAZON FEE,1,12/7/2010 15:08,13541.33,NaN,United Kingdom
135534,547901,AMAZONFEE,AMAZON FEE,1,3/28/2011 11:57,219.76,NaN,United Kingdom


## StockCode with S

In [19]:
getInfoStockCode('S')

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
14436,C537581,S,SAMPLES,-1,12/7/2010 12:03,12.95,NaN,United Kingdom
14437,C537581,S,SAMPLES,-1,12/7/2010 12:03,52.00,NaN,United Kingdom
96680,C544580,S,SAMPLES,-1,2/21/2011 14:25,5.74,NaN,United Kingdom
96681,C544580,S,SAMPLES,-1,2/21/2011 14:25,11.08,NaN,United Kingdom
96682,C544580,S,SAMPLES,-1,2/21/2011 14:25,5.79,NaN,United Kingdom


Unique StockCodes matching 'S': 63 out of 541909 (0.01% of dataset)
Breakdown:
  * 0 have UnitPrice equal to 0
  * 0 have negative UnitPrice
  * 0 have Quantity equal to 0
  * 61 have negative Quantity
  * 63 have null CustomerID
  * 61 have an InvoiceNo starting with 'C' (Cancellations/Returns)


In [20]:
df[(df['StockCode'] == 'S') & (df['Quantity'] > 0)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
152709,549684,S,SAMPLES,1,4/11/2011 13:24,30.00,NaN,United Kingdom
419666,572849,S,SAMPLES,1,10/26/2011 12:20,33.05,NaN,United Kingdom


## Quantity

In [21]:
print(f"Rows with Quantity = 0: {len(df[df['Quantity'] == 0])} ({len(df[df['Quantity'] == 0]) / len(df) * 100:.2f}%)")
print(f"Rows with Quantity < 0: {len(df[df['Quantity'] < 0])} ({len(df[df['Quantity'] < 0]) / len(df) * 100:.2f}%)")

neg_qty = df[df['Quantity'] < 0]
print(f"Rows with Quantity = -1: {len(neg_qty[neg_qty['Quantity'] == -1])} out of negative quantities ({len(neg_qty[neg_qty['Quantity'] == -1]) / len(neg_qty) * 100:.2f}%)")

Rows with Quantity = 0: 0 (0.00%)
Rows with Quantity < 0: 10624 (1.96%)
Rows with Quantity = -1: 4184 out of negative quantities (39.38%)


## UnitPrice

In [22]:
print(f"Rows with UnitPrice = 0: {len(df[df['UnitPrice'] == 0])} ({len(df[df['UnitPrice'] == 0]) / len(df) * 100:.2f}%)")
print(f"Rows with UnitPrice < 0: {len(df[df['UnitPrice'] < 0])} ({len(df[df['UnitPrice'] < 0]) / len(df) * 100:.2f}%)")

Rows with UnitPrice = 0: 2515 (0.46%)
Rows with UnitPrice < 0: 2 (0.00%)


## CustomerID

In [23]:
print(f"Rows with missing CustomerID: {len(df[df['CustomerID'].isnull()])}")

Rows with missing CustomerID: 135080


## Executive Summary of Data Discovery

**1. The Nature of Special Codes (Costs & Returns)**
The profiling reveals a critical pattern: operational codes such as `D` and `CRUK` exist almost exclusively with negative quantities. The same applies to the vast majority of `S` and `AMAZONFEE` records. 
* **Conclusion**: With the exception of `POST` and `DOT`, these special codes represent cash outflows, discounts, or write-offs. 
* **ETL Action**: By strictly enforcing the rule `Quantity > 0` in the pipeline, the automated script inherently purges all discounts, charity commissions, and passive fees, sanitizing the dataset without requiring complex, hardcoded rules for each specific `StockCode`.

**2. The Background Noise of Zero-Pricing**
The analysis surfaced 2,515 rows where `UnitPrice = 0`. In e-commerce environments, these typically represent system tests, free gifts (like `S` samples at zero price), or database errors. Furthermore, the only records with prices strictly less than zero are the two `B` accounting adjustments.
* **Conclusion**: A transaction with a zero price generates neither revenue nor loss; it is statistical noise that degrades query performance.
* **ETL Action**: Enforcing `UnitPrice > 0` simultaneously drops all 2,515 zero-value artifacts and the 2 accounting errors, ensuring the Data Warehouse is populated strictly with economically viable transactions.

**3. The CustomerID Dilemma (Preserving 25% of Revenue)**
The data reveals that 135,080 rows lack a `CustomerID`. This accounts for roughly 25% of the entire dataset.
* **Conclusion**: Executing a standard `dropna()` on this column would obliterate a quarter of the valid store receipts, severely skewing global revenue analytics and compromising the Fact Table's integrity. These represent "Guest Checkouts".
* **ETL Action**: Implementing `df['CustomerID'].fillna(0)` is the architecturally sound solution. It consolidates all unregistered purchases under a dedicated surrogate ID (0), fully preserving the revenue stream for high-level dashboarding.

**Final Verdict**
The exploratory evidence gathered in this notebook validates the data quality logic: filling null customers with `0` and enforcing strict bounds (`Quantity > 0` and `UnitPrice > 0`) creates an elegant, highly effective ETL pipeline that isolates pure business transactions.